# KNN desde cero con ventanas

En este notebook se implementará manualmente un clasificador K-Nearest Neighbors (KNN). NumPy se utilizará para las operaciones matemáticas y Pandas para leer los archivos, pero no se utilizará una implementación de modelos de Scikit-learn.

La versión ejecutable independiente se encuentra en `modelo_knn_ventanas_guzman.py`.

## Librerías necesarias

Solamente se utilizarán herramientas para localizar archivos, cargar las tablas y realizar operaciones numéricas.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Carga de los conjuntos preparados

El ETL generó dos archivos: `datos_train.csv` y `datos_test.csv`. Cada fila contiene 120 características estadísticas y la columna `actividad` como etiqueta.

Se cargarán ambas tablas y posteriormente se separarán las características en `X` y las etiquetas en `y`.

In [2]:
# Ruta de la carpeta donde se guardaron los datos.
RUTA_DATOS_MODELO = Path("../datos_modelo")

# Cargamos las tablas y conservamos la etiqueta como texto para
# mantener actividades como 000, 001, 002, etc.
datos_train = pd.read_csv(
    RUTA_DATOS_MODELO / "datos_train.csv",
    dtype={"actividad": str}
)

datos_test = pd.read_csv(
    RUTA_DATOS_MODELO / "datos_test.csv",
    dtype={"actividad": str}
)

# X contiene solamente las características numéricas.
X_train = datos_train.drop(columns="actividad").to_numpy()
X_test = datos_test.drop(columns="actividad").to_numpy()

# y contiene la actividad correspondiente a cada fila.
y_train = datos_train["actividad"].to_numpy()
y_test = datos_test["actividad"].to_numpy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (12448, 120)
y_train: (12448,)
X_test: (3112, 120)
y_test: (3112,)


### Interpretación

El conjunto de entrenamiento contiene 12,448 observaciones y el conjunto de prueba 3,112. Cada observación está representada por 120 características y posee una etiqueta de actividad correspondiente.

Las características y etiquetas tienen la misma cantidad de filas en cada conjunto, por lo que se encuentran correctamente alineadas.

## 2. Estandarización de las características

KNN clasifica una observación según la distancia con los datos de entrenamiento. Por esta razón, las características deben encontrarse en escalas comparables.

La media y la desviación estándar se calcularán únicamente con `X_train`. Estos mismos parámetros se aplicarán a `X_test` para evitar utilizar información del conjunto de prueba durante la preparación.

In [3]:
# Media y desviación estándar de cada característica,
# calculadas solamente con entrenamiento.
media_train = np.mean(X_train, axis=0)
desviacion_train = np.std(X_train, axis=0)

# Evitamos una división entre cero si alguna característica
# permanece constante.
desviacion_train[desviacion_train == 0] = 1

X_train_escalado = (
    (X_train - media_train) / desviacion_train
).astype(np.float64)

X_test_escalado = (
    (X_test - media_train) / desviacion_train
).astype(np.float64)

print("X_train escalado:", X_train_escalado.shape)
print("X_test escalado:", X_test_escalado.shape)
print("Valores faltantes en train:", np.isnan(X_train_escalado).sum())
print("Valores faltantes en test:", np.isnan(X_test_escalado).sum())

X_train escalado: (12448, 120)
X_test escalado: (3112, 120)
Valores faltantes en train: 0
Valores faltantes en test: 0


### Interpretación

Los conjuntos conservaron sus dimensiones, pero ahora las características se encuentran en escalas comparables. La transformación utilizó exclusivamente los parámetros calculados con entrenamiento y no generó valores faltantes.

## 3. Implementación manual de KNN

KNN clasifica una observación buscando los ejemplos de entrenamiento más cercanos. La implementación realizará los siguientes pasos:

1. Guardar los datos y etiquetas de entrenamiento.
2. Calcular la distancia euclidiana con las observaciones de prueba.
3. Seleccionar los cinco vecinos más cercanos.
4. Contar las etiquetas de los vecinos.
5. Asignar la actividad con mayor cantidad de votos.

Los datos se procesarán por bloques para controlar el uso de memoria.

In [4]:
class KNNDesdeCero:
    """Clasificador K-Nearest Neighbors implementado con NumPy."""

    def __init__(self, k=5, tamanio_bloque=64):
        self.k = k
        self.tamanio_bloque = tamanio_bloque

    def fit(self, X, y):
        """Guarda los datos y las etiquetas de entrenamiento."""

        if len(X) != len(y):
            raise ValueError(
                "X y y deben tener la misma cantidad de filas."
            )

        if self.k < 1 or self.k > len(X):
            raise ValueError(
                "k debe estar entre 1 y la cantidad de muestras."
            )

        self.X_train = np.asarray(X, dtype=np.float64)

        # Convertimos las etiquetas a números para facilitar
        # el conteo de votos.
        self.clases, self.y_numerico = np.unique(
            y,
            return_inverse=True
        )

        self.norma_train = np.sum(
            self.X_train ** 2,
            axis=1
        )

        return self

    def _predecir_bloque(self, X_bloque):
        """Predice un grupo pequeño de observaciones."""

        # Distancia euclidiana al cuadrado:
        # ||a-b||² = ||a||² + ||b||² - 2(a·b)
        norma_prueba = np.sum(
            X_bloque ** 2,
            axis=1,
            keepdims=True
        )

        with np.errstate(
            over="ignore",
            divide="ignore",
            invalid="ignore"
        ):
            distancias = (
                norma_prueba
                + self.norma_train
                - 2 * X_bloque @ self.X_train.T
            )

        distancias = np.maximum(distancias, 0)

        # Obtenemos los índices de los k vecinos con menor distancia.
        indices_vecinos = np.argpartition(
            distancias,
            kth=self.k - 1,
            axis=1
        )[:, :self.k]

        predicciones = []

        for fila, vecinos in enumerate(indices_vecinos):
            etiquetas_vecinos = self.y_numerico[vecinos]

            votos = np.bincount(
                etiquetas_vecinos,
                minlength=len(self.clases)
            )

            clases_ganadoras = np.flatnonzero(
                votos == votos.max()
            )

            if len(clases_ganadoras) == 1:
                clase_elegida = clases_ganadoras[0]
            else:
                # En un empate gana la clase empatada cuyo
                # vecino se encuentre más cerca.
                orden = np.argsort(
                    distancias[fila, vecinos]
                )

                clase_elegida = next(
                    etiquetas_vecinos[posicion]
                    for posicion in orden
                    if etiquetas_vecinos[posicion]
                    in clases_ganadoras
                )

            predicciones.append(
                self.clases[clase_elegida]
            )

        return np.asarray(predicciones)

    def predict(self, X):
        """Genera predicciones procesando los datos por bloques."""

        X = np.asarray(X, dtype=np.float64)
        predicciones = []

        for inicio in range(
            0,
            len(X),
            self.tamanio_bloque
        ):
            fin = inicio + self.tamanio_bloque

            predicciones.extend(
                self._predecir_bloque(X[inicio:fin])
            )

        return np.asarray(predicciones)

## 4. Entrenamiento y evaluación

Se utilizará `k=5`, por lo que cada predicción dependerá de las cinco observaciones de entrenamiento más cercanas. Posteriormente, todas las predicciones se compararán directamente con las etiquetas reales del conjunto de prueba.

In [5]:
# Creamos y entrenamos el modelo manual.
modelo_knn = KNNDesdeCero(
    k=5,
    tamanio_bloque=64
)

modelo_knn.fit(
    X_train_escalado,
    y_train
)

print("Datos de entrenamiento almacenados.")

# Generamos una predicción para cada fila de prueba.
y_pred_knn = modelo_knn.predict(
    X_test_escalado
)

print("Cantidad de predicciones:", len(y_pred_knn))

print("\nPrimeras 10 predicciones:")
for posicion in range(10):
    print(
        f"Observación {posicion + 1}: "
        f"real={y_test[posicion]}, "
        f"predicha={y_pred_knn[posicion]}"
    )

# Calculamos manualmente la exactitud.
predicciones_correctas = np.sum(
    y_pred_knn == y_test
)

exactitud_knn = (
    predicciones_correctas / len(y_test)
)

print("\nPredicciones correctas:", predicciones_correctas)
print("Cantidad total:", len(y_test))
print(f"Exactitud: {exactitud_knn:.4f}")
print(f"Porcentaje de aciertos: {exactitud_knn * 100:.2f}%")

Datos de entrenamiento almacenados.
Cantidad de predicciones: 3112

Primeras 10 predicciones:
Observación 1: real=006, predicha=006
Observación 2: real=013, predicha=013
Observación 3: real=007, predicha=007
Observación 4: real=014, predicha=014
Observación 5: real=006, predicha=012
Observación 6: real=014, predicha=014
Observación 7: real=007, predicha=007
Observación 8: real=005, predicha=005
Observación 9: real=015, predicha=015
Observación 10: real=007, predicha=007

Predicciones correctas: 2818
Cantidad total: 3112
Exactitud: 0.9055
Porcentaje de aciertos: 90.55%


### Interpretación

El modelo KNN implementado desde cero clasificó correctamente **2,818 de las 3,112 observaciones** del conjunto de prueba, obteniendo una exactitud de **90.55%**. Esto significa que aproximadamente 91 de cada 100 observaciones fueron asignadas a la actividad correcta.

La evaluación se realizó directamente sobre las filas independientes generadas por el ETL. No se efectuaron agrupaciones ni combinaciones posteriores de predicciones. El resultado corresponde a la configuración inicial de KNN con `k=5`.

## 5. Métricas por actividad y matriz de confusión

La exactitud resume el porcentaje total de aciertos, pero no muestra cómo se comporta el modelo en cada actividad. Por ello se calcularán manualmente precision, recall, F1-score y soporte para cada una de las 16 clases.

También se construirá una matriz de confusión. Sus filas representan las actividades reales y sus columnas las actividades predichas. Los valores de la diagonal son clasificaciones correctas y los valores fuera de ella muestran las confusiones entre actividades.

In [6]:
def calcular_metricas_clasificacion(y_real, y_predicha):
    """Calcula manualmente las métricas y la matriz de confusión."""

    clases = np.unique(
        np.concatenate([y_real, y_predicha])
    )

    posicion_clase = {
        clase: posicion
        for posicion, clase in enumerate(clases)
    }

    # Filas: actividad real. Columnas: actividad predicha.
    matriz = np.zeros(
        (len(clases), len(clases)),
        dtype=int
    )

    for real, predicha in zip(y_real, y_predicha):
        matriz[
            posicion_clase[real],
            posicion_clase[predicha]
        ] += 1

    verdaderos_positivos = np.diag(matriz).astype(float)
    falsos_positivos = matriz.sum(axis=0) - verdaderos_positivos
    falsos_negativos = matriz.sum(axis=1) - verdaderos_positivos
    soporte = matriz.sum(axis=1)

    precision = np.divide(
        verdaderos_positivos,
        verdaderos_positivos + falsos_positivos,
        out=np.zeros_like(verdaderos_positivos),
        where=(verdaderos_positivos + falsos_positivos) != 0
    )

    recall = np.divide(
        verdaderos_positivos,
        verdaderos_positivos + falsos_negativos,
        out=np.zeros_like(verdaderos_positivos),
        where=(verdaderos_positivos + falsos_negativos) != 0
    )

    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) != 0
    )

    reporte = pd.DataFrame(
        {
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": soporte
        },
        index=clases
    )

    promedios = {
        "precision_macro": np.mean(precision),
        "recall_macro": np.mean(recall),
        "f1_macro": np.mean(f1),
        "f1_ponderado": np.average(f1, weights=soporte)
    }

    return clases, matriz, reporte, promedios

In [7]:
# Calculamos las métricas con las predicciones de prueba.
clases, matriz_confusion, reporte_metricas, promedios = (
    calcular_metricas_clasificacion(
        y_test,
        y_pred_knn
    )
)

print("Métricas por actividad:")
display(reporte_metricas.round(4))

print("Promedios generales:")
print(f"Precision macro: {promedios['precision_macro']:.4f}")
print(f"Recall macro: {promedios['recall_macro']:.4f}")
print(f"F1-score macro: {promedios['f1_macro']:.4f}")
print(f"F1-score ponderado: {promedios['f1_ponderado']:.4f}")

# Convertimos la matriz en una tabla con nombres claros.
matriz_confusion_df = pd.DataFrame(
    matriz_confusion,
    index=[f"Real {clase}" for clase in clases],
    columns=[f"Pred {clase}" for clase in clases]
)

print("\nMatriz de confusión:")
display(matriz_confusion_df)

Métricas por actividad:


,precision,recall,f1_score,support
000,0.9855,0.9577,0.9714,142
001,0.8846,0.9020,0.8932,153
002,0.8485,0.8235,0.8358,136
003,0.9080,0.9250,0.9164,160
004,0.9431,0.9005,0.9213,221
005,0.8963,0.9600,0.9270,225
006,0.8235,0.8284,0.8260,169
007,0.9169,0.9739,0.9445,306
008,0.9483,0.9205,0.9342,239
009,0.9215,0.9065,0.9139,246


Promedios generales:
Precision macro: 0.9022
Recall macro: 0.9003
F1-score macro: 0.9009
F1-score ponderado: 0.9053

Matriz de confusión:


,Pred 000,Pred 001,Pred 002,Pred 003,Pred 004,Pred 005,Pred 006,Pred 007,Pred 008,Pred 009,Pred 010,Pred 011,Pred 012,Pred 013,Pred 014,Pred 015
Real 000,136,0,2,2,0,0,0,0,2,0,0,0,0,0,0,0
Real 001,1,138,6,4,0,1,0,1,0,0,1,1,0,0,0,0
Real 002,0,8,112,7,0,5,1,0,0,1,1,0,1,0,0,0
Real 003,1,3,4,148,0,0,0,1,0,1,0,2,0,0,0,0
Real 004,0,0,0,0,199,12,1,2,0,0,2,2,1,0,0,2
Real 005,0,0,1,0,4,216,1,1,0,0,0,2,0,0,0,0
Real 006,0,1,1,1,0,2,140,1,0,0,0,0,17,2,0,4
Real 007,0,0,0,0,2,0,0,298,1,0,0,2,2,1,0,0
Real 008,0,0,0,1,0,0,0,2,220,10,0,1,0,0,0,5
Real 009,0,1,4,0,0,0,1,5,7,223,2,1,0,0,1,1


### Interpretación de las métricas

El modelo obtuvo un F1-score macro de **0.9009** y un F1-score ponderado de **0.9053**. La cercanía entre ambos valores y la exactitud de **90.55%** indica que el desempeño general no depende solamente de las actividades con mayor cantidad de observaciones.

Las actividades con mejor F1-score fueron `000` (**0.9714**), `007` (**0.9445**), `010` (**0.9351**) y `008` (**0.9342**). La actividad con mayor dificultad fue `012`, con recall de **0.7578** y F1-score de **0.7722**. También presentaron resultados inferiores al promedio las actividades `006` y `002`, con F1-scores de **0.8260** y **0.8358**.

La matriz de confusión permite observar que los errores más frecuentes incluyen la clasificación de observaciones de `012` como `006`, de `013` como `014` y de `006` como `012`. Estos resultados sugieren que dichas actividades presentan características estadísticas similares para el clasificador KNN.

## Implementación ejecutable

La versión final se encuentra en `modelo_knn_ventanas_guzman.py`. Este archivo contiene la misma carga de datos, estandarización, implementación manual de KNN, predicción y evaluación, y puede ejecutarse directamente desde una terminal sin depender de Jupyter Notebook.